# Question 28: Build a Complete LLM Inference Engine (SOLUTION)

### Difficulty: Expert

### Problem Statement

In this capstone exercise, you will build a **minimal but complete** LLM inference engine that integrates several components you have learned:

1. **A simple transformer model** (2-layer, with self-attention and feed-forward layers)
2. **KV cache** for efficient autoregressive generation
3. **Top-p (nucleus) sampling** for controllable text generation
4. **Basic batching** to handle multiple concurrent requests (prompts of different lengths share one batch)

The engine accepts text prompts (using a simple character-level tokenizer), generates completions, and supports multiple concurrent requests.

This mirrors the core of what production inference engines (vLLM, TGI, TensorRT-LLM) do, stripped down to its essentials.

---

### Requirements

1. **`SimpleTokenizer`**: Character-level tokenizer.
   - `encode(text) -> List[int]`: Convert text to token IDs.
   - `decode(ids) -> str`: Convert token IDs back to text.
   - Reserve ID 0 for `<pad>` and ID 1 for `<eos>`; characters are `ord(c) + 2` (ASCII only).

2. **`MiniTransformer(nn.Module)`**: A 2-layer transformer.
   - Token embedding + positional embedding.
   - Each layer: multi-head self-attention (with KV cache support) + feed-forward network.
   - Layer normalization (pre-norm style).
   - Output projection to vocabulary logits.
   - `forward(input_ids, kv_caches=None, attention_mask=None, position_ids=None)`: with caches, `input_ids` holds only the new tokens — a whole prompt (prefill) or one token per row (decode). `attention_mask` is a `(B, cached + new)` boolean padding mask (True = real token) and `position_ids` gives each new token's position; by default positions continue from the cache length.

3. **`InferenceEngine`**: Ties everything together.
   - `generate(prompts, max_new_tokens, temperature, top_p)`: Accept a list of text prompts, tokenize, generate completions, decode back to text.
   - Uses KV cache internally for efficient generation: one prefill pass over all prompts, then one cached decode pass per new token.
   - Implements top-p sampling.
   - Batches prompts of different lengths **without changing any result**: left-pad them, mask the padding out of attention (the pad keys stay in the KV cache, so the mask grows with every decode step), and count positions over real tokens only.

4. **Validation**: Generate text from multiple prompts simultaneously, and check (with greedy decoding) that every prompt gets exactly the completion it gets on its own.

---

### Constraints

- Use only PyTorch operations.
- The transformer must use pre-norm (LayerNorm before attention/FFN, not after).
- Top-p sampling: sort probabilities descending, compute cumulative sum, zero out tokens beyond the cumulative threshold `p`, renormalize, then sample.
- Batched generation must be equivalent to one-prompt-at-a-time generation: with greedy decoding, `generate([a, b])` equals `[generate([a])[0], generate([b])[0]]`.
- The model does NOT need to produce coherent text (it is untrained). The goal is correct architecture and inference logic.

---

### Company Tags

Perplexity, Together AI, Anyscale, Fireworks AI

---

<details>
  <summary>Hint</summary>

  - For the tokenizer, use `ord(c) + 2` as the token ID for character `c` (offset by 2 to reserve 0 and 1 for special tokens). For decoding, `chr(id - 2)`.
  - For KV cache in the transformer: each layer gets its own KVCache. Pass a list of caches (one per layer) through the forward pass.
  - For top-p sampling: `sorted_probs, sorted_indices = torch.sort(probs, descending=True)`. Then `cumsum = torch.cumsum(sorted_probs, dim=-1)`. Mask out where `cumsum - sorted_probs >= top_p`.
  - For batching: **left**-pad all prompts to the same length, so every row's last prompt token sits in the last column and each decode step appends one column for all rows. (Right padding would put each row's next token at a different column.)
  - Combine the causal mask with a key-padding mask `(B, kv_len)` so no query attends to a pad key. A pad *query* must still attend to something (e.g. itself): a softmax row of all `-inf` is NaN, and `0 * NaN` leaks it into real positions in the next layer.
  - Position ids per row: `attention_mask.cumsum(-1) - 1` for the prompt, then `prompt_len + step` while decoding — so a padded prompt sees exactly the positions it would see alone.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import List, Optional, Tuple

In [ ]:
# Configuration
torch.manual_seed(42)

d_model = 64
num_heads = 4
d_ff = 128              # Feed-forward hidden dimension
num_layers = 2
max_seq_len = 256
temperature = 0.8
top_p = 0.9

device = "cpu"
print(f"Config: d_model={d_model}, heads={num_heads}, layers={num_layers}, d_ff={d_ff}")

In [ ]:
class SimpleTokenizer:
    """
    Character-level tokenizer.
    
    Special tokens:
        0 = <pad>
        1 = <eos>
    Regular characters: ord(c) + 2
    """
    
    def __init__(self):
        self.pad_id = 0
        self.eos_id = 1
        # Cover all 128 ASCII code points + 2 special tokens
        self.vocab_size = 128 + 2  # 130 total
    
    def encode(self, text):
        """Convert text to list of token IDs."""
        ids = [ord(c) + 2 for c in text]
        if any(i >= self.vocab_size for i in ids):
            raise ValueError("SimpleTokenizer only covers ASCII characters")
        return ids
    
    def decode(self, ids):
        """Convert list of token IDs back to text. Skip pad and eos."""
        chars = []
        for token_id in ids:
            if token_id in (self.pad_id, self.eos_id):  # control ids, not characters
                continue
            chars.append(chr(token_id - 2))
        return ''.join(chars)


# Quick test
tok = SimpleTokenizer()
assert tok.decode(tok.encode("abc")) == "abc"
print(f"SimpleTokenizer: vocab_size={tok.vocab_size}, pad_id={tok.pad_id}, eos_id={tok.eos_id}")

In [ ]:
class KVCache:
    """KV Cache for a single attention layer: K and V of shape (B, num_heads, seq_len, d_head)."""
    
    def __init__(self):
        self.k = None
        self.v = None
    
    def update(self, new_k, new_v):
        """Append new K/V along the sequence axis and return full cached K/V."""
        if self.k is None:
            self.k = new_k
            self.v = new_v
        else:
            self.k = torch.cat([self.k, new_k], dim=2)
            self.v = torch.cat([self.v, new_v], dim=2)
        return self.k, self.v
    
    @property
    def seq_len(self):
        """Return the current cached sequence length."""
        return 0 if self.k is None else self.k.shape[2]
    
    def reset(self):
        self.k = None
        self.v = None


class MultiHeadAttention(nn.Module):
    """Multi-head attention with KV cache support and an optional key-padding mask."""
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, kv_cache=None, attention_mask=None):
        """
        Args:
            x: (B, S, d_model) -- the new token(s)
            kv_cache: Optional KVCache
            attention_mask: Optional (B, kv_len) bool, True for real tokens and False for
                padding, covering the cached positions followed by the S new ones
        Returns:
            output: (B, S, d_model)
        """
        B, S, _ = x.shape
        
        q = self.W_q(x).view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        k = self.W_k(x).view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        
        if kv_cache is not None:
            k, v = kv_cache.update(k, v)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, H, S, kv_len)
        
        # Causal mask, offset by the cached length: new token i sits at absolute position
        # past_len + i and may attend to keys 0..past_len + i. This also covers a multi-token
        # PREFILL that passes a cache; a single decode token may attend to all of the past.
        kv_len = k.shape[2]
        past_len = kv_len - S
        allowed = torch.ones(S, kv_len, dtype=torch.bool, device=x.device).tril(diagonal=past_len)
        allowed = allowed.expand(B, S, kv_len)
        if attention_mask is not None:
            # No query may attend to a padding key. A padding query still attends to itself,
            # so its (ignored) row is never all -inf -- that would softmax to NaN, and the
            # NaN would leak into real positions through 0 * NaN in the next layer.
            is_self = torch.arange(kv_len, device=x.device) == (past_len + torch.arange(S, device=x.device)).unsqueeze(1)
            allowed = allowed & (attention_mask[:, None, :] | is_self)
        scores = scores.masked_fill(~allowed.unsqueeze(1), float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, v)
        
        output = output.transpose(1, 2).contiguous().view(B, S, self.d_model)
        return self.W_o(output)


class FeedForward(nn.Module):
    """Simple feed-forward network: Linear -> GELU -> Linear."""
    
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    """Single transformer block: pre-norm attention + pre-norm FFN."""
    
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
    
    def forward(self, x, kv_cache=None, attention_mask=None):
        """
        Pre-norm transformer block:
            x = x + Attention(LayerNorm(x))
            x = x + FFN(LayerNorm(x))
        """
        x = x + self.attn(self.ln1(x), kv_cache=kv_cache, attention_mask=attention_mask)
        x = x + self.ffn(self.ln2(x))
        return x


class MiniTransformer(nn.Module):
    """
    Minimal 2-layer transformer for text generation.
    
    Components:
    - Token embedding + positional embedding
    - N transformer blocks (each with attention + FFN)
    - Final layer norm
    - Output projection to vocab logits
    """
    
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        
        self.final_norm = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, input_ids, kv_caches=None, attention_mask=None, position_ids=None):
        """
        Args:
            input_ids: (B, S) token indices -- the new tokens
            kv_caches: Optional list of KVCache (one per layer)
            attention_mask: Optional (B, cached + S) bool, False marks padding
            position_ids: Optional (B, S) positions of the new tokens. Defaults to
                cache_len .. cache_len + S - 1 (the same for every row)
        Returns:
            logits: (B, S, vocab_size)
        """
        B, S = input_ids.shape
        
        if position_ids is None:
            # The caches are updated inside attention, so at this point they still hold
            # only the past: the new tokens start at position cache.seq_len.
            past_len = kv_caches[0].seq_len if kv_caches is not None else 0
            position_ids = torch.arange(past_len, past_len + S, device=input_ids.device).expand(B, S)
        
        # Token + positional embeddings
        x = self.token_emb(input_ids) + self.pos_emb(position_ids)
        
        # Pass through transformer blocks
        for i, block in enumerate(self.blocks):
            cache = kv_caches[i] if kv_caches is not None else None
            x = block(x, kv_cache=cache, attention_mask=attention_mask)
        
        # Final norm + output projection
        x = self.final_norm(x)
        logits = self.output_proj(x)
        
        return logits


print("Model components defined.")

In [ ]:
def top_p_sample(logits, temperature=1.0, top_p=0.9):
    """
    Top-p (nucleus) sampling.
    
    Args:
        logits: (B, vocab_size) raw logits for next token
        temperature: Sampling temperature
        top_p: Cumulative probability threshold
    
    Returns:
        sampled_tokens: (B, 1) sampled token IDs
    """
    # 1. Apply temperature
    scaled_logits = logits / temperature
    
    # 2. Softmax
    probs = F.softmax(scaled_logits, dim=-1)  # (B, vocab_size)
    
    # 3. Sort descending
    sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
    
    # 4. Cumulative sum
    cumsum = torch.cumsum(sorted_probs, dim=-1)
    
    # 5. Create mask: drop a token when the mass ranked above it already reaches top_p.
    #    This keeps the smallest set of tokens whose cumulative prob >= top_p
    #    (including the token that crosses top_p), and always the top token.
    mask = (cumsum - sorted_probs) >= top_p  # True for tokens to REMOVE
    mask[..., 0] = False
    
    # 6. Zero out masked tokens
    sorted_probs = sorted_probs.masked_fill(mask, 0.0)
    
    # 7. Renormalize
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
    
    # 8. Sample from the filtered distribution
    sampled_sorted_idx = torch.multinomial(sorted_probs, num_samples=1)  # (B, 1)
    
    # 9. Map back to original token IDs
    sampled_tokens = sorted_indices.gather(dim=-1, index=sampled_sorted_idx)  # (B, 1)
    
    return sampled_tokens


class InferenceEngine:
    """
    Complete inference engine tying together:
    - Tokenizer
    - Transformer model with KV cache
    - Top-p sampling
    - Basic (static) batching of prompts with different lengths
    """
    
    def __init__(self, model, tokenizer, device="cpu"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
    
    @torch.no_grad()
    def generate(self, prompts, max_new_tokens=50, temperature=0.8, top_p=0.9):
        """
        Generate completions for a list of text prompts.
        
        Args:
            prompts: List of text strings (non-empty)
            max_new_tokens: Maximum tokens to generate
            temperature: Sampling temperature
            top_p: Nucleus sampling threshold
        
        Returns:
            List of generated text strings (prompt + completion)
        """
        batch_size = len(prompts)
        
        # 1. Tokenize all prompts
        encoded = [self.tokenizer.encode(p) for p in prompts]
        max_prompt_len = max(len(e) for e in encoded)
        
        # 2. LEFT-pad to the same length, so every row's last prompt token sits in the
        #    last column and each decode step appends all rows' new tokens in one column.
        #    The mask marks real tokens; pads must never be attended to (they also live in
        #    the KV cache), and positions count real tokens only, so a padded prompt sees
        #    exactly the positions it would see on its own.
        padded = [[self.tokenizer.pad_id] * (max_prompt_len - len(e)) + e for e in encoded]
        input_ids = torch.tensor(padded, device=self.device)                # (B, max_prompt_len)
        attention_mask = input_ids.new_tensor(
            [[0] * (max_prompt_len - len(e)) + [1] * len(e) for e in encoded]).bool()
        position_ids = (attention_mask.long().cumsum(dim=-1) - 1).clamp(min=0)
        
        # 3. Create KV caches (one per layer) -- fresh for every call
        kv_caches = [KVCache() for _ in range(self.model.num_layers)]
        
        # 4. Prefill: run model on all prompts at once to populate the KV caches
        logits = self.model(input_ids, kv_caches=kv_caches,
                            attention_mask=attention_mask, position_ids=position_ids)
        next_logits = logits[:, -1, :]                       # last REAL token of every row
        next_positions = attention_mask.sum(dim=-1, keepdim=True)  # (B, 1) = prompt lengths
        
        # 5. Decode loop
        all_generated = [[] for _ in range(batch_size)]
        done = [False] * batch_size
        
        for step in range(max_new_tokens):
            # Sample next token using top-p
            next_tokens = top_p_sample(next_logits, temperature=temperature, top_p=top_p)  # (B, 1)
            
            # Record generated tokens
            for i in range(batch_size):
                if not done[i]:
                    token = next_tokens[i].item()
                    if token == self.tokenizer.eos_id:
                        done[i] = True
                    else:
                        all_generated[i].append(token)
            
            # Stop when every sequence is finished or the budget is used up
            if all(done) or step == max_new_tokens - 1:
                break
            
            # Feed the new tokens through the model with the KV cache. Finished rows keep
            # decoding (their output is ignored) so the batch stays rectangular.
            attention_mask = torch.cat(
                [attention_mask, attention_mask.new_ones(batch_size, 1)], dim=1)
            logits = self.model(next_tokens, kv_caches=kv_caches,
                                attention_mask=attention_mask, position_ids=next_positions)
            next_positions = next_positions + 1
            next_logits = logits[:, -1, :]  # (B, vocab_size)
        
        # 6. Decode back to text
        outputs = []
        for i in range(batch_size):
            generated_text = self.tokenizer.decode(all_generated[i])
            outputs.append(prompts[i] + generated_text)
        
        return outputs


print("InferenceEngine defined.")

In [ ]:
# ============================================================
# Validation: Generate from multiple prompts
# ============================================================

torch.manual_seed(42)

# Initialize tokenizer
tokenizer = SimpleTokenizer()
print(f"Vocab size: {tokenizer.vocab_size}")

# Test tokenizer roundtrip
test_text = "Hello, World!"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"Tokenizer roundtrip: '{test_text}' -> {encoded} -> '{decoded}'")
assert decoded == test_text, "Tokenizer roundtrip failed!"
print()

# Initialize model and engine
model = MiniTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    max_seq_len=max_seq_len
).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print()

engine = InferenceEngine(model, tokenizer, device=device)

# Generate from multiple prompts
prompts = [
    "The quick brown ",
    "In a galaxy ",
    "def fibonacci(",
    "Once upon a ",
]

print("Generating completions...")
print("=" * 60)

with torch.no_grad():
    outputs = engine.generate(
        prompts,
        max_new_tokens=40,
        temperature=temperature,
        top_p=top_p
    )

for i, (prompt, output) in enumerate(zip(prompts, outputs)):
    generated_part = output[len(prompt):]
    print(f"\nPrompt {i}: '{prompt}'")
    print(f"Generated: '{generated_part}'")
    print(f"Full: '{output}'")

# Batching must not change results: with (near-)greedy decoding, every prompt in the
# batch must get exactly the completion it gets when generated on its own.
with torch.no_grad():
    batched = engine.generate(prompts, max_new_tokens=20, temperature=1e-4, top_p=1.0)
    alone = [engine.generate([p], max_new_tokens=20, temperature=1e-4, top_p=1.0)[0] for p in prompts]
for p, b, a in zip(prompts, batched, alone):
    assert b == a, f"Prompt {p!r}: batched {b!r} != alone {a!r} -- check padding mask / positions"
print("\nBatched generation matches one-prompt-at-a-time generation for every prompt.")

print("\n" + "=" * 60)
print("Inference engine validation complete.")
print("(Note: output is random since the model is untrained.)")